# Spark–Iceberg 카탈로그 연결 점검

Spark 세션이 `lakehouse` Iceberg REST Catalog를 사용할 수 있는지
최소한의 읽기 전용 쿼리로 확인합니다.

**보여주는 역량:** 인프라 연결 진단, Spark 설정 확인, 빠른 실패 감지

## Goal

- Spark의 기본 카탈로그 설정을 확인합니다.
- REST Catalog의 네임스페이스와 테이블 목록을 조회합니다.
- 메달리온 네임스페이스가 준비됐는지 검증합니다.

## Setup

In [1]:
from pyspark.sql import SparkSession

CATALOG = "lakehouse"
EXPECTED_NAMESPACES = {"bronze", "silver", "gold"}

spark = (
    SparkSession.builder
    .appName("spark-catalog-smoke-test")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

## Steps

### 1. 핵심 설정 확인

In [2]:
config_keys = [
    "spark.sql.defaultCatalog",
    "spark.sql.catalog.lakehouse",
    "spark.sql.catalog.lakehouse.type",
    "spark.sql.catalog.lakehouse.uri",
    "spark.sql.catalog.lakehouse.warehouse",
]

for key in config_keys:
    print(f"{key} = {spark.conf.get(key)}")

spark.sql.defaultCatalog = lakehouse
spark.sql.catalog.lakehouse = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.lakehouse.type = rest
spark.sql.catalog.lakehouse.uri = http://iceberg-rest:8181
spark.sql.catalog.lakehouse.warehouse = s3://warehouse/


### 2. 카탈로그 객체 조회

In [3]:
namespace_df = spark.sql(
    f"SHOW NAMESPACES IN {CATALOG}"
)
namespace_df.show(truncate=False)

for namespace in ("bronze", "silver", "gold"):
    print(f"\n[{CATALOG}.{namespace}]")
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{namespace}"
    ).show(truncate=False)

+---------+
|namespace|
+---------+
|bronze   |
|silver   |
|gold     |
+---------+


[lakehouse.bronze]
+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|bronze   |daily_prices_raw |false      |
|bronze   |minute_prices_raw|false      |
+---------+-----------------+-----------+


[lakehouse.silver]
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|silver   |daily_prices |false      |
|silver   |minute_prices|false      |
+---------+-------------+-----------+


[lakehouse.gold]


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



## Checks

In [4]:
actual_namespaces = {
    row.namespace for row in namespace_df.collect()
}
missing_namespaces = EXPECTED_NAMESPACES - actual_namespaces

assert not missing_namespaces, (
    "namespace_admin.ipynb를 먼저 실행하세요. "
    f"누락: {sorted(missing_namespaces)}"
)
print("Spark–Iceberg REST Catalog 연결 확인 완료")

Spark–Iceberg REST Catalog 연결 확인 완료


## Next Steps

연결 검증이 끝나면 `pipelines/medallion.ipynb`를 실행해
Bronze/Silver 테이블과 변환 흐름을 확인합니다.